# Smartphones domain

Domain-specific pools, templates, generation functions, and query builders.

In [1]:
# Load common helpers only if this domain notebook is run standalone.
# Robust loader supports both .py and .ipynb helper layouts used across domains.
import json
from pathlib import Path as _Path

if "BenchmarkExample" not in globals():
    helper_candidates = [
        _Path("common_helpers.py"),
        _Path("../common_helpers.py"),
        _Path("00_common_helpers.ipynb"),
        _Path("../00_common_helpers.ipynb"),
    ]
    loaded = False
    for hp in helper_candidates:
        if not hp.exists():
            continue
        if hp.suffix == ".py":
            exec(hp.read_text(encoding="utf-8"), globals())
            loaded = True
            break
        if hp.suffix == ".ipynb":
            _nb = json.loads(hp.read_text(encoding="utf-8"))
            for _cell in _nb.get("cells", []):
                if _cell.get("cell_type") != "code":
                    continue
                _src = "".join(_cell.get("source", []))
                _stripped = _src.strip()
                if not _stripped or _stripped.startswith("!pip") or _stripped.startswith("%run"):
                    continue
                exec(compile(_src, str(hp), "exec"), globals())
            loaded = True
            break
    if not loaded:
        raise FileNotFoundError("Cannot find common_helpers.py or 00_common_helpers.ipynb")


✅ Patched: WikidataClient.sparql_select (robust) + load_or_build_pool (safe)
✅ Patched: select_items_with_* используют ru/en fallback + repair_pool_labels чинит QID вместо label


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<a id="smartphones-domain"></a>

## 22. Smartphones domain

Smartphone templates with manufacturer, operating system, release year, screen, and CPU-style constraints.

In [ ]:
import pandas as pd
import random
import datetime as dt
from collections import defaultdict
from typing import Optional, List, Tuple, Dict, Any

Q_SMARTPHONE = ensure_qid("смартфон", fallback_qid="Q22645")            # smartphone
Q_MOBILE_PHONE = ensure_qid("мобильный телефон", fallback_qid="Q17517")  # mobile phone
Q_ANDROID = ensure_qid("Android", fallback_qid="Q94")
Q_IOS = ensure_qid("iOS", fallback_qid="Q48493")

Q_SMARTPHONE_MODEL = resolve_qid("модель смартфона", "ru") or resolve_qid("smartphone model", "en")
Q_MOBILE_PHONE_MODEL = resolve_qid("модель мобильного телефона", "ru") or resolve_qid("mobile phone model", "en")

SMARTPHONE_TYPE_QIDS: List[str] = [q for q in [
    Q_SMARTPHONE,
    Q_MOBILE_PHONE,
    Q_SMARTPHONE_MODEL,
    Q_MOBILE_PHONE_MODEL,
] if q]

SMARTPHONE_MAKER_ANCHORS = [
    ("Q312", "Apple"),
    ("Q27414", "Samsung"),
    ("Q174187", "Xiaomi"),
    ("Q175751", "Huawei"),
    ("Q95", "Google"),
    ("Q318144", "Sony"),
    ("Q3884", "Nokia"),
    ("Q207194", "Motorola"),
    ("Q215380", "OnePlus"),
    ("Q1134006", "Honor"),
    ("Q1078460", "Oppo"),
    ("Q679694", "Vivo"),
    ("Q257998", "LG"),
    ("Q15148", "HTC"),
    ("Q192608", "Lenovo"),
]
SMARTPHONE_OS_ANCHORS = [
    (Q_ANDROID, "Android"),
    (Q_IOS, "iOS"),
]

_LABEL_CACHE: Dict[str, str] = {}
_SMARTPHONE_ENUM_CACHE: Dict[str, List[Any]] = {}
_SMARTPHONE_ENUM_CURSOR: Dict[str, int] = defaultdict(int)
_SMARTPHONE_CANDIDATE_CACHE: Dict[Tuple[Any, ...], List[Any]] = {}

def _norm_lbl(x: Any) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    s = str(x).strip()
    return "" if s.lower() == "nan" else s

def get_label_ru_en_cached(qid: str) -> str:
    qid = (qid or "").strip()
    if not qid or not qid.startswith("Q"):
        return ""
    if qid in _LABEL_CACHE:
        return _LABEL_CACHE[qid]

    query = f"""
SELECT ?ru ?en WHERE {{
  OPTIONAL {{ wd:{qid} rdfs:label ?ru FILTER(LANG(?ru)='ru') }}
  OPTIONAL {{ wd:{qid} rdfs:label ?en FILTER(LANG(?en)='en') }}
}}
LIMIT 1
"""
    lbl = ""
    try:
        rows = rows_from_select(wd.sparql_select(query))
        if rows:
            lbl = _norm_lbl(rows[0].get("ru")) or _norm_lbl(rows[0].get("en"))
    except Exception:
        lbl = ""
    _LABEL_CACHE[qid] = lbl
    return lbl

def _smartphone_type_block(item_var: str = "p") -> str:
    values = " ".join(f"wd:{q}" for q in SMARTPHONE_TYPE_QIDS if q)
    return f"""
  ?{item_var} wdt:P31/wdt:P279* ?phoneClass .
  VALUES ?phoneClass {{ {values} }}
  OPTIONAL {{ ?{item_var} wdt:P306 ?_anyOs . }}
  FILTER(
    ?phoneClass = wd:{Q_SMARTPHONE}
    {"|| ?phoneClass = wd:" + Q_SMARTPHONE_MODEL if Q_SMARTPHONE_MODEL else ""}
    || BOUND(?_anyOs)
  )
"""

def _smartphone_date_block(item_var: str = "p") -> List[str]:
    return [
        f"OPTIONAL {{ ?{item_var} wdt:P577 ?d1 . }}",
        f"OPTIONAL {{ ?{item_var} wdt:P571 ?d2 . }}",
        "BIND(COALESCE(?d1, ?d2) AS ?d)",
        "FILTER(BOUND(?d))",
    ]

def _smartphone_label_block(item_var: str = "p") -> str:
    return f"""
  OPTIONAL {{ ?{item_var} rdfs:label ?ru FILTER(LANG(?ru)='ru') }}
  OPTIONAL {{ ?{item_var} rdfs:label ?en FILTER(LANG(?en)='en') }}
  BIND(COALESCE(?ru, ?en) AS ?label)
"""

def _smartphone_gold_limit(k: int, complexity: str) -> int:
    if complexity in ("L4", "L5"):
        return max(50, k * 20)
    return max(80, k * 25)

def _smartphone_items_from_rows(rows: List[Dict[str, Any]], q_key: str = "p", lbl_key: str = "label") -> List[Tuple[str, str]]:
    out: List[Tuple[str, str]] = []
    seen = set()
    for r in rows:
        qid = uri_to_qid(r.get(q_key, ""))
        lbl = _norm_lbl(r.get(lbl_key))
        if not qid or qid in seen:
            continue
        seen.add(qid)
        if not lbl:
            lbl = get_label_ru_en_cached(qid)
        if qid:
            out.append((qid, lbl))
    return out

def run_smartphones_query(
    maker_qid: Optional[str] = None,
    os_qid: Optional[str] = None,
    year_from: Optional[int] = None,
    year_to: Optional[int] = None,
    not_maker_qid: Optional[str] = None,
    limit: int = 120,
) -> Tuple[str, List[Tuple[str, str]], List[str]]:

    where_lines: List[str] = [
        _smartphone_type_block("p"),
        "?p wdt:P176 ?maker .",
        "?p wikibase:sitelinks ?sl .",
        "FILTER(?sl >= 1)",
    ]

    if maker_qid:
        where_lines.append(f"?p wdt:P176 wd:{maker_qid} .")
    if os_qid:
        where_lines.append(f"?p wdt:P306 wd:{os_qid} .")
    if not_maker_qid:
        where_lines.append(f"FILTER NOT EXISTS {{ ?p wdt:P176 wd:{not_maker_qid} . }}")

    if year_from is not None or year_to is not None:
        where_lines.extend(_smartphone_date_block("p"))
        if year_from is not None:
            where_lines.append(f"FILTER(YEAR(?d) >= {int(year_from)})")
        if year_to is not None:
            where_lines.append(f"FILTER(YEAR(?d) <= {int(year_to)})")

    where = "\n  ".join(where_lines)
    sparql = f"""
SELECT DISTINCT ?p ?label WHERE {{
  {where}
  {_smartphone_label_block("p")}
}}
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    items = _smartphone_items_from_rows(rows, q_key="p", lbl_key="label")
    return sparql.strip(), items, where_lines

def _next_candidate(key: str, builder, rng: random.Random):
    if key not in _SMARTPHONE_ENUM_CACHE:
        items = list(builder() or [])
        rng.shuffle(items)
        _SMARTPHONE_ENUM_CACHE[key] = items
        _SMARTPHONE_ENUM_CURSOR[key] = 0

    items = _SMARTPHONE_ENUM_CACHE[key]
    if not items:
        return None

    cur = _SMARTPHONE_ENUM_CURSOR.get(key, 0)
    if cur >= len(items):
        if len(items) > 1:
            rng.shuffle(items)
        cur = 0
    _SMARTPHONE_ENUM_CURSOR[key] = cur + 1
    return items[cur]

def _fallback_maker_candidates() -> List[Tuple[str, str, int]]:
    out = []
    for qid, lbl in SMARTPHONE_MAKER_ANCHORS:
        out.append((qid, lbl or get_label_ru_en_cached(qid) or "", 999))
    return out

def _fallback_maker_os_candidates() -> List[Tuple[str, str, str, str, int]]:
    out = []
    for maker_qid, maker_lbl in SMARTPHONE_MAKER_ANCHORS:
        for os_qid, os_lbl in SMARTPHONE_OS_ANCHORS:
            out.append((maker_qid, maker_lbl or get_label_ru_en_cached(maker_qid) or "",
                        os_qid, os_lbl or get_label_ru_en_cached(os_qid) or "", 999))
    return out

def find_smartphone_maker_candidates(min_models: int = 5, limit: int = 80) -> List[Tuple[str, str, int]]:
    key = ("makers", int(min_models), int(limit))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    sparql = f"""
SELECT ?maker ?makerLabel (COUNT(DISTINCT ?p) AS ?cnt) WHERE {{
  {_smartphone_type_block("p")}
  ?p wdt:P176 ?maker .
  ?p wikibase:sitelinks ?sl .
  FILTER(?sl >= 1)
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
}}
GROUP BY ?maker ?makerLabel
HAVING(COUNT(DISTINCT ?p) >= {int(min_models)})
ORDER BY DESC(COUNT(DISTINCT ?p))
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    out: List[Tuple[str, str, int]] = []
    for r in rows:
        maker_qid = uri_to_qid(r.get("maker", ""))
        maker_lbl = _norm_lbl(r.get("makerLabel")) or (get_label_ru_en_cached(maker_qid) if maker_qid else "")
        try:
            cnt = int(float(r.get("cnt", "0")))
        except Exception:
            cnt = 0
        if maker_qid and maker_lbl and cnt >= int(min_models):
            out.append((maker_qid, maker_lbl, cnt))

    if not out:
        out = _fallback_maker_candidates()

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out

def find_smartphone_maker_year_candidates(
    min_models: int = 5,
    year_from: int = 2010,
    year_to: int = 2025,
    limit: int = 240,
) -> List[Tuple[str, str, int, int]]:
    key = ("maker_year", int(min_models), int(year_from), int(year_to), int(limit))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    sparql = f"""
SELECT ?maker ?makerLabel ?yy (COUNT(DISTINCT ?p) AS ?cnt) WHERE {{
  {_smartphone_type_block("p")}
  ?p wdt:P176 ?maker .
  {' '.join(_smartphone_date_block("p"))}
  BIND(YEAR(?d) AS ?yy)
  FILTER(?yy >= {int(year_from)} && ?yy <= {int(year_to)})
  ?p wikibase:sitelinks ?sl .
  FILTER(?sl >= 1)
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
}}
GROUP BY ?maker ?makerLabel ?yy
HAVING(COUNT(DISTINCT ?p) >= {int(min_models)})
ORDER BY DESC(COUNT(DISTINCT ?p))
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    out: List[Tuple[str, str, int, int]] = []
    for r in rows:
        maker_qid = uri_to_qid(r.get("maker", ""))
        maker_lbl = _norm_lbl(r.get("makerLabel")) or (get_label_ru_en_cached(maker_qid) if maker_qid else "")
        try:
            yy = int(float(r.get("yy", "0")))
            cnt = int(float(r.get("cnt", "0")))
        except Exception:
            yy, cnt = 0, 0
        if maker_qid and maker_lbl and yy and cnt >= int(min_models):
            out.append((maker_qid, maker_lbl, yy, cnt))

    if not out:
        years = list(range(2018, 2025))
        makers = find_smartphone_maker_candidates(min_models=max(2, min_models - 2), limit=20)
        for maker_qid, maker_lbl, _ in makers:
            for yy in years:
                out.append((maker_qid, maker_lbl, yy, max(min_models, 5)))

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out

def find_smartphone_maker_os_candidates(min_models: int = 5, limit: int = 160) -> List[Tuple[str, str, str, str, int]]:
    key = ("maker_os", int(min_models), int(limit))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    sparql = f"""
SELECT ?maker ?makerLabel ?os ?osLabel (COUNT(DISTINCT ?p) AS ?cnt) WHERE {{
  {_smartphone_type_block("p")}
  ?p wdt:P176 ?maker ;
     wdt:P306 ?os .
  ?p wikibase:sitelinks ?sl .
  FILTER(?sl >= 1)
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
}}
GROUP BY ?maker ?makerLabel ?os ?osLabel
HAVING(COUNT(DISTINCT ?p) >= {int(min_models)})
ORDER BY DESC(COUNT(DISTINCT ?p))
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    out: List[Tuple[str, str, str, str, int]] = []
    for r in rows:
        maker_qid = uri_to_qid(r.get("maker", ""))
        os_qid = uri_to_qid(r.get("os", ""))
        maker_lbl = _norm_lbl(r.get("makerLabel")) or (get_label_ru_en_cached(maker_qid) if maker_qid else "")
        os_lbl = _norm_lbl(r.get("osLabel")) or (get_label_ru_en_cached(os_qid) if os_qid else "")
        try:
            cnt = int(float(r.get("cnt", "0")))
        except Exception:
            cnt = 0
        if maker_qid and os_qid and maker_lbl and os_lbl and cnt >= int(min_models):
            out.append((maker_qid, maker_lbl, os_qid, os_lbl, cnt))

    if not out:
        out = _fallback_maker_os_candidates()

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out

def find_smartphone_maker_os_year_candidates(
    min_models: int = 1,
    year_from: int = 2010,
    year_to: int = 2025,
    limit: int = 320,
) -> List[Tuple[str, str, str, str, int, int]]:
    key = ("maker_os_year", int(min_models), int(year_from), int(year_to), int(limit))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    sparql = f"""
SELECT ?maker ?makerLabel ?os ?osLabel ?yy (COUNT(DISTINCT ?p) AS ?cnt) WHERE {{
  {_smartphone_type_block("p")}
  ?p wdt:P176 ?maker ;
     wdt:P306 ?os .
  {' '.join(_smartphone_date_block("p"))}
  BIND(YEAR(?d) AS ?yy)
  FILTER(?yy >= {int(year_from)} && ?yy <= {int(year_to)})
  ?p wikibase:sitelinks ?sl .
  FILTER(?sl >= 1)
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
}}
GROUP BY ?maker ?makerLabel ?os ?osLabel ?yy
HAVING(COUNT(DISTINCT ?p) >= {int(min_models)})
ORDER BY DESC(COUNT(DISTINCT ?p))
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    out: List[Tuple[str, str, str, str, int, int]] = []
    for r in rows:
        maker_qid = uri_to_qid(r.get("maker", ""))
        os_qid = uri_to_qid(r.get("os", ""))
        maker_lbl = _norm_lbl(r.get("makerLabel")) or (get_label_ru_en_cached(maker_qid) if maker_qid else "")
        os_lbl = _norm_lbl(r.get("osLabel")) or (get_label_ru_en_cached(os_qid) if os_qid else "")
        try:
            yy = int(float(r.get("yy", "0")))
            cnt = int(float(r.get("cnt", "0")))
        except Exception:
            yy, cnt = 0, 0
        if maker_qid and os_qid and maker_lbl and os_lbl and yy and cnt >= int(min_models):
            out.append((maker_qid, maker_lbl, os_qid, os_lbl, yy, cnt))

    if not out:
        base = find_smartphone_maker_os_candidates(min_models=max(2, min_models), limit=40)
        for maker_qid, maker_lbl, os_qid, os_lbl, _ in base:
            for yy in range(2018, 2025):
                out.append((maker_qid, maker_lbl, os_qid, os_lbl, yy, max(min_models, 1)))

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out

def build_smartphone_l5_candidates(
    min_models: int = 1,
    year_from: int = 2010,
    year_to: int = 2025,
    base_limit: int = 120,
    not_makers_per_base: int = 4,
) -> List[Tuple[str, str, str, str, int, str, str, int]]:
    key = ("l5", int(min_models), int(year_from), int(year_to), int(base_limit), int(not_makers_per_base))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    base = find_smartphone_maker_os_year_candidates(
        min_models=min_models,
        year_from=year_from,
        year_to=year_to,
        limit=base_limit,
    )
    makers = find_smartphone_maker_candidates(min_models=max(2, min_models), limit=40)

    out: List[Tuple[str, str, str, str, int, str, str, int]] = []
    for maker_qid, maker_lbl, os_qid, os_lbl, yy, cnt in base:
        others = [(q, l) for q, l, _ in makers if q != maker_qid][:max(1, int(not_makers_per_base))]
        for not_maker_qid, not_maker_lbl in others:
            out.append((maker_qid, maker_lbl, os_qid, os_lbl, yy, not_maker_qid, not_maker_lbl, cnt))

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out


_SMARTPHONE_MODELS_BY_MAKER_CACHE: Dict[Tuple[str, int, int, int], List[Dict[str, Any]]] = {}

def _smartphone_date_block_named(
    item_var: str = "p",
    date_var: str = "d",
    d1_var: str = "d1",
    d2_var: str = "d2",
) -> List[str]:
    return [
        f"OPTIONAL {{ ?{item_var} wdt:P577 ?{d1_var} . }}",
        f"OPTIONAL {{ ?{item_var} wdt:P571 ?{d2_var} . }}",
        f"BIND(COALESCE(?{d1_var}, ?{d2_var}) AS ?{date_var})",
        f"FILTER(BOUND(?{date_var}))",
    ]


def fetch_smartphone_models_for_maker(
    maker_qid: str,
    min_year: int = 2007,
    max_year: int = 2026,
    limit: int = 300,
) -> List[Dict[str, Any]]:
    key = (maker_qid, int(min_year), int(max_year), int(limit))
    if key in _SMARTPHONE_MODELS_BY_MAKER_CACHE:
        return _SMARTPHONE_MODELS_BY_MAKER_CACHE[key]

    sparql = f"""
SELECT DISTINCT ?p ?label ?d WHERE {{
  {_smartphone_type_block("p")}
  ?p wdt:P176 wd:{maker_qid} .
  ?p wikibase:sitelinks ?sl .
  FILTER(?sl >= 1)
  {' '.join(_smartphone_date_block_named("p", "d", "d1m", "d2m"))}
  FILTER(YEAR(?d) >= {int(min_year)} && YEAR(?d) <= {int(max_year)})
  {_smartphone_label_block("p")}
}}
ORDER BY ASC(?d) ?label
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))

    out: List[Dict[str, Any]] = []
    seen: set = set()
    for r in rows:
        qid = uri_to_qid(r.get("p", ""))
        if not qid or qid in seen:
            continue
        seen.add(qid)

        lbl = _norm_lbl(r.get("label")) or get_label_ru_en_cached(qid)
        d = _norm_lbl(r.get("d"))
        m = re.match(r"^(\d{4})", d)
        if not m:
            continue
        yy = int(m.group(1))
        if yy < int(min_year) or yy > int(max_year):
            continue
        out.append({
            "qid": qid,
            "label": lbl,
            "date": d,
            "year": yy,
        })

    _SMARTPHONE_MODELS_BY_MAKER_CACHE[key] = out
    return out


def find_smartphone_maker_year_window_candidates(
    min_models: int = 5,
    maker_limit: int = 24,
    min_span_years: int = 2,
    max_span_years: int = 4,
    max_candidates_per_maker: int = 12,
) -> List[Tuple[str, str, int, int, int]]:
    key = (
        "maker_year_window",
        int(min_models),
        int(maker_limit),
        int(min_span_years),
        int(max_span_years),
        int(max_candidates_per_maker),
    )
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    makers = find_smartphone_maker_candidates(min_models=max(min_models + 1, 6), limit=maker_limit)
    out: List[Tuple[str, str, int, int, int]] = []

    for maker_qid, maker_lbl, _ in makers:
        models = fetch_smartphone_models_for_maker(maker_qid)
        if len(models) < min_models:
            continue

        year_counts: Dict[int, int] = defaultdict(int)
        for m in models:
            year_counts[int(m["year"])] += 1

        years = sorted(year_counts)
        local: List[Tuple[str, str, int, int, int]] = []
        for i, y1 in enumerate(years):
            total = 0
            for j in range(i, len(years)):
                y2 = years[j]
                total += int(year_counts[y2])
                span = y2 - y1 + 1
                if span < int(min_span_years):
                    continue
                if span > int(max_span_years):
                    break
                if total >= int(min_models):
                    local.append((maker_qid, maker_lbl, y1, y2, total))

        local.sort(key=lambda x: (abs((x[3] - x[2] + 1) - 3), abs(x[4] - 7), x[2], x[3]))
        out.extend(local[:max_candidates_per_maker])

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out


def find_smartphone_maker_threshold_year_candidates(
    min_models: int = 5,
    maker_limit: int = 24,
    max_candidates_per_maker: int = 12,
) -> List[Tuple[str, str, str, int, int]]:
    key = ("maker_threshold_year", int(min_models), int(maker_limit), int(max_candidates_per_maker))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    makers = find_smartphone_maker_candidates(min_models=max(min_models + 1, 6), limit=maker_limit)
    out: List[Tuple[str, str, str, int, int]] = []

    for maker_qid, maker_lbl, _ in makers:
        models = fetch_smartphone_models_for_maker(maker_qid)
        years = sorted(int(m["year"]) for m in models)
        uniq = sorted(set(years))
        if len(uniq) < 3:
            continue

        local: List[Tuple[str, str, str, int, int]] = []
        for yy in uniq[1:-1]:
            cnt_ge = sum(1 for y in years if y >= yy)
            cnt_lt = sum(1 for y in years if y < yy)
            cnt_le = sum(1 for y in years if y <= yy)
            cnt_gt = sum(1 for y in years if y > yy)

            if cnt_ge >= int(min_models) and cnt_lt >= 1:
                local.append((maker_qid, maker_lbl, "from", yy, cnt_ge))
            if cnt_le >= int(min_models) and cnt_gt >= 1:
                local.append((maker_qid, maker_lbl, "to", yy, cnt_le))

        local.sort(key=lambda x: (abs(x[4] - 8), x[3], 0 if x[2] == "from" else 1))
        out.extend(local[:max_candidates_per_maker])

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out


def run_smartphones_query_relative_model(
    maker_qid: str,
    ref_model_qid: str,
    relation: str = "after",
    limit: int = 120,
) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    if relation not in {"after", "before"}:
        raise ValueError(f"Unsupported relation: {relation}")

    where_lines: List[str] = [
        _smartphone_type_block("p"),
        f"?p wdt:P176 wd:{maker_qid} .",
        "?p wikibase:sitelinks ?sl .",
        "FILTER(?sl >= 1)",
        *_smartphone_date_block_named("p", "d", "d1p", "d2p"),
        f"VALUES ?refModel {{ wd:{ref_model_qid} }}",
        *_smartphone_date_block_named("refModel", "refDate", "d1ref", "d2ref"),
        "FILTER(?p != ?refModel)",
    ]
    if relation == "after":
        where_lines.append("FILTER(?d > ?refDate)")
    else:
        where_lines.append("FILTER(?d < ?refDate)")

    where = "\n  ".join(where_lines)
    sparql = f"""
SELECT DISTINCT ?p ?label WHERE {{
  {where}
  {_smartphone_label_block("p")}
}}
ORDER BY ASC(?label)
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    items = _smartphone_items_from_rows(rows, q_key="p", lbl_key="label")
    return sparql.strip(), items, where_lines


def run_smartphones_query_between_models(
    maker_qid: str,
    left_model_qid: str,
    right_model_qid: str,
    limit: int = 120,
) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    where_lines: List[str] = [
        _smartphone_type_block("p"),
        f"?p wdt:P176 wd:{maker_qid} .",
        "?p wikibase:sitelinks ?sl .",
        "FILTER(?sl >= 1)",
        *_smartphone_date_block_named("p", "d", "d1p", "d2p"),
        f"VALUES ?leftRef {{ wd:{left_model_qid} }}",
        *_smartphone_date_block_named("leftRef", "leftDate", "d1l", "d2l"),
        f"VALUES ?rightRef {{ wd:{right_model_qid} }}",
        *_smartphone_date_block_named("rightRef", "rightDate", "d1r", "d2r"),
        "FILTER(?p != ?leftRef && ?p != ?rightRef)",
        "FILTER(?leftDate < ?rightDate)",
        "FILTER(?d > ?leftDate && ?d < ?rightDate)",
    ]

    where = "\n  ".join(where_lines)
    sparql = f"""
SELECT DISTINCT ?p ?label WHERE {{
  {where}
  {_smartphone_label_block("p")}
}}
ORDER BY ASC(?label)
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    items = _smartphone_items_from_rows(rows, q_key="p", lbl_key="label")
    return sparql.strip(), items, where_lines


def find_smartphone_reference_comparison_candidates(
    min_models: int = 5,
    maker_limit: int = 20,
    max_candidates_per_maker: int = 12,
) -> List[Tuple[str, str, str, str, int, str, int]]:
    key = ("ref_compare", int(min_models), int(maker_limit), int(max_candidates_per_maker))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    makers = find_smartphone_maker_candidates(min_models=max(min_models + 2, 7), limit=maker_limit)
    out: List[Tuple[str, str, str, str, int, str, int]] = []

    for maker_qid, maker_lbl, _ in makers:
        models = fetch_smartphone_models_for_maker(maker_qid)
        n = len(models)
        if n < min_models + 2:
            continue

        local: List[Tuple[str, str, str, str, int, str, int]] = []
        for i, m in enumerate(models):
            ref_qid = m["qid"]
            ref_lbl = _norm_lbl(m["label"])
            ref_year = int(m["year"])

            cnt_after = max(0, n - i - 1)
            cnt_before = max(0, i)

            if cnt_after >= int(min_models) and cnt_before >= 1 and ref_lbl:
                local.append((maker_qid, maker_lbl, ref_qid, ref_lbl, ref_year, "after", cnt_after))
            if cnt_before >= int(min_models) and cnt_after >= 1 and ref_lbl:
                local.append((maker_qid, maker_lbl, ref_qid, ref_lbl, ref_year, "before", cnt_before))

        local.sort(
            key=lambda x: (
                abs(x[6] - 8),
                abs(x[4] - 2018),
                0 if x[5] == "after" else 1,
            )
        )
        out.extend(local[:max_candidates_per_maker])

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out


def find_smartphone_between_model_candidates(
    min_models_between: int = 2,
    maker_limit: int = 18,
    max_candidates_per_maker: int = 10,
) -> List[Tuple[str, str, str, str, int, str, str, int, int]]:
    key = ("between_models", int(min_models_between), int(maker_limit), int(max_candidates_per_maker))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    makers = find_smartphone_maker_candidates(min_models=max(min_models_between + 5, 7), limit=maker_limit)
    out: List[Tuple[str, str, str, str, int, str, str, int, int]] = []

    for maker_qid, maker_lbl, _ in makers:
        models = fetch_smartphone_models_for_maker(maker_qid)
        n = len(models)
        if n < min_models_between + 3:
            continue

        local: List[Tuple[str, str, str, str, int, str, str, int, int]] = []
        for i in range(n):
            left = models[i]
            for j in range(i + min_models_between + 1, n):
                right = models[j]
                between = j - i - 1
                if between < int(min_models_between):
                    continue

                year_gap = int(right["year"]) - int(left["year"])
                if year_gap < 1:
                    continue

                local.append((
                    maker_qid,
                    maker_lbl,
                    left["qid"],
                    _norm_lbl(left["label"]),
                    int(left["year"]),
                    right["qid"],
                    _norm_lbl(right["label"]),
                    int(right["year"]),
                    between,
                ))

        local.sort(
            key=lambda x: (
                abs(x[8] - 4),
                abs((x[7] - x[4]) - 4),
                x[4],
                x[7],
            )
        )
        out.extend(local[:max_candidates_per_maker])

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out

def _smartphones_now_iso() -> str:
    try:
        return utc_now_z()
    except Exception:
        return dt.datetime.now(dt.timezone.utc).isoformat().replace("+00:00", "Z")

def _smartphone_make_example(
    idx: int,
    complexity: str,
    query_text_ru: str,
    constraints: Dict[str, Any],
    items: List[Tuple[str, str]],
    sparql_query: str,
    template_id: str,
) -> BenchmarkExample:
    return BenchmarkExample(
        id=f"smartphones_{complexity.lower()}_{idx:05d}",
        domain="smartphones",
        complexity=complexity,
        query_text_ru=query_text_ru,
        constraints=constraints,
        requested_count=5,
        gold_answer_qids=[q for q, _ in items],
        gold_answer_labels_ru=[l for _, l in items],
        sparql_query=sparql_query,
        created_at=_smartphones_now_iso(),
        is_advanced=False,
        template_id=template_id,
        template_family="default",
        gold_truncated=False,
        ask_validator_sparql=None,
    )


def _generate_smartphones_example_default(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 40,
) -> BenchmarkExample:
    k = 5

    if complexity == "L1":
        for _ in range(max_attempts):
            cand = _next_candidate(
                "smartphones_L1",
                lambda: find_smartphone_maker_candidates(min_models=k, limit=120),
                rng,
            )
            if not cand:
                break
            maker_qid, maker_lbl, _ = cand
            sparql, items, _ = run_smartphones_query(
                maker_qid=maker_qid,
                limit=_smartphone_gold_limit(k, complexity),
            )
            if len(items) < k:
                continue
            return _smartphone_make_example(
                idx=idx,
                complexity=complexity,
                query_text_ru=f"Назови {k} моделей смартфонов производителя «{maker_lbl}».",
                constraints={
                    "maker_qid": maker_qid,
                    "maker_label_ru": maker_lbl,
                },
                items=items,
                sparql_query=sparql,
                template_id="smartphones_maker_only",
            )
        raise RuntimeError("Smartphones L1: no valid maker candidates")

    if complexity == "L2":
        for _ in range(max_attempts):
            cand = _next_candidate(
                "smartphones_L2",
                lambda: find_smartphone_maker_year_window_candidates(
                    min_models=k,
                    maker_limit=24,
                    min_span_years=2,
                    max_span_years=4,
                    max_candidates_per_maker=12,
                ),
                rng,
            )
            if not cand:
                break
            maker_qid, maker_lbl, y1, y2, _ = cand
            sparql, items, _ = run_smartphones_query(
                maker_qid=maker_qid,
                year_from=y1,
                year_to=y2,
                limit=_smartphone_gold_limit(k, complexity),
            )
            if len(items) < k:
                continue
            return _smartphone_make_example(
                idx=idx,
                complexity=complexity,
                query_text_ru=f"Назови {k} смартфонов производителя «{maker_lbl}», выпущенных в период {y1}–{y2} годов.",
                constraints={
                    "maker_qid": maker_qid,
                    "maker_label_ru": maker_lbl,
                    "year_from": y1,
                    "year_to": y2,
                },
                items=items,
                sparql_query=sparql,
                template_id="smartphones_maker_year_window",
            )
        raise RuntimeError("Smartphones L2: no valid maker+year-window candidates")

    if complexity == "L3":
        for _ in range(max_attempts):
            cand = _next_candidate(
                "smartphones_L3",
                lambda: find_smartphone_maker_threshold_year_candidates(
                    min_models=k,
                    maker_limit=24,
                    max_candidates_per_maker=12,
                ),
                rng,
            )
            if not cand:
                break
            maker_qid, maker_lbl, mode, yy, _ = cand
            if mode == "from":
                sparql, items, _ = run_smartphones_query(
                    maker_qid=maker_qid,
                    year_from=yy,
                    limit=_smartphone_gold_limit(k, complexity),
                )
                query_text_ru = f"Назови {k} смартфонов производителя «{maker_lbl}», выпущенных не раньше {yy} года."
                constraints = {
                    "maker_qid": maker_qid,
                    "maker_label_ru": maker_lbl,
                    "year_from": yy,
                    "year_mode": "from",
                }
                template_id = "smartphones_maker_year_from"
            else:
                sparql, items, _ = run_smartphones_query(
                    maker_qid=maker_qid,
                    year_to=yy,
                    limit=_smartphone_gold_limit(k, complexity),
                )
                query_text_ru = f"Назови {k} смартфонов производителя «{maker_lbl}», выпущенных не позже {yy} года."
                constraints = {
                    "maker_qid": maker_qid,
                    "maker_label_ru": maker_lbl,
                    "year_to": yy,
                    "year_mode": "to",
                }
                template_id = "smartphones_maker_year_to"

            if len(items) < k:
                continue
            return _smartphone_make_example(
                idx=idx,
                complexity=complexity,
                query_text_ru=query_text_ru,
                constraints=constraints,
                items=items,
                sparql_query=sparql,
                template_id=template_id,
            )
        raise RuntimeError("Smartphones L3: no valid maker+threshold-year candidates")

    if complexity == "L4":
        for _ in range(max_attempts):
            cand = _next_candidate(
                "smartphones_L4",
                lambda: find_smartphone_reference_comparison_candidates(
                    min_models=k,
                    maker_limit=20,
                    max_candidates_per_maker=12,
                ),
                rng,
            )
            if not cand:
                break
            maker_qid, maker_lbl, ref_qid, ref_lbl, ref_year, relation, _ = cand
            sparql, items, _ = run_smartphones_query_relative_model(
                maker_qid=maker_qid,
                ref_model_qid=ref_qid,
                relation=relation,
                limit=_smartphone_gold_limit(k, complexity),
            )
            if len(items) < k:
                continue

            if relation == "after":
                query_text_ru = (
                    f"Назови {k} смартфонов производителя «{maker_lbl}», "
                    f"выпущенных позже модели «{ref_lbl}» ({ref_year})."
                )
                template_id = "smartphones_maker_after_model"
            else:
                query_text_ru = (
                    f"Назови {k} смартфонов производителя «{maker_lbl}», "
                    f"выпущенных раньше модели «{ref_lbl}» ({ref_year})."
                )
                template_id = "smartphones_maker_before_model"

            return _smartphone_make_example(
                idx=idx,
                complexity=complexity,
                query_text_ru=query_text_ru,
                constraints={
                    "maker_qid": maker_qid,
                    "maker_label_ru": maker_lbl,
                    "reference_model_qid": ref_qid,
                    "reference_model_label_ru": ref_lbl,
                    "reference_model_year": ref_year,
                    "relation": relation,
                },
                items=items,
                sparql_query=sparql,
                template_id=template_id,
            )
        raise RuntimeError("Smartphones L4: no valid maker+reference-model candidates")

    if complexity == "L5":
        for _ in range(max_attempts):
            cand = _next_candidate(
                "smartphones_L5",
                lambda: find_smartphone_between_model_candidates(
                    min_models_between=2,
                    maker_limit=18,
                    max_candidates_per_maker=10,
                ),
                rng,
            )
            if not cand:
                break
            maker_qid, maker_lbl, left_qid, left_lbl, left_year, right_qid, right_lbl, right_year, _ = cand
            sparql, items, _ = run_smartphones_query_between_models(
                maker_qid=maker_qid,
                left_model_qid=left_qid,
                right_model_qid=right_qid,
                limit=_smartphone_gold_limit(k, complexity),
            )
            if len(items) < 1:
                continue
            return _smartphone_make_example(
                idx=idx,
                complexity=complexity,
                query_text_ru=(
                    f"Подбери до {k} смартфонов производителя «{maker_lbl}», "
                    f"выпущенных позже модели «{left_lbl}» ({left_year}), "
                    f"но раньше модели «{right_lbl}» ({right_year})."
                ),
                constraints={
                    "maker_qid": maker_qid,
                    "maker_label_ru": maker_lbl,
                    "left_reference_model_qid": left_qid,
                    "left_reference_model_label_ru": left_lbl,
                    "left_reference_model_year": left_year,
                    "right_reference_model_qid": right_qid,
                    "right_reference_model_label_ru": right_lbl,
                    "right_reference_model_year": right_year,
                    "relation": "between_models",
                },
                items=items,
                sparql_query=sparql,
                template_id="smartphones_maker_between_models",
            )
        raise RuntimeError("Smartphones L5: no valid maker+between-models candidates")

    raise ValueError(f"Unknown complexity: {complexity}")


def generate_smartphones_example_advanced(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 40,
) -> BenchmarkExample:
    return _generate_smartphones_example_default(complexity, idx, rng, max_attempts=max_attempts)

def generate_smartphones_example(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 40,
) -> BenchmarkExample:
    return _generate_smartphones_example_default(complexity, idx, rng, max_attempts=max_attempts)


## Patched high-quality multihop smartphones generator

Overrides the starter generator with rich-schema WDQS-only templates for L1–L5.

In [ ]:
# ============================================================
# PATCHED SMARTPHONES GENERATOR: clean WDQS-only multihop tasks
# ============================================================
# This cell intentionally overrides generate_smartphones_example(_advanced)
# from the lightweight starter cell above. It keeps the same public API used by
# notebook_runner.py, but emits records in the same rich JSON schema as the
# curated domains: RU+EN query text, RU+EN gold labels, clean English
# constraints, ASK validator, local_validator and gold_collection_meta.

import re
import json
import random
import datetime as _dt
from collections import Counter, defaultdict
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

DOMAIN = "smartphones"

SMARTPHONES_WDQS_LIMIT_DEFAULT = int(globals().get("SMARTPHONES_WDQS_LIMIT_DEFAULT", 1200))
SMARTPHONES_WDQS_HARD_TIMEOUT_SECONDS = int(globals().get("SMARTPHONES_WDQS_HARD_TIMEOUT_SECONDS", 35))
SMARTPHONES_STRICT_FULL_GOLD = bool(globals().get("SMARTPHONES_STRICT_FULL_GOLD", True))
SMARTPHONES_MIN_GOLD_BY_LEVEL = dict(globals().get("SMARTPHONES_MIN_GOLD_BY_LEVEL", {
    "L1": 6,
    "L2": 6,
    "L3": 6,
    "L4": 5,
    "L5": 3,
}))
SMARTPHONES_MAX_GOLD_BY_LEVEL = dict(globals().get("SMARTPHONES_MAX_GOLD_BY_LEVEL", {
    "L1": 500,
    "L2": 500,
    "L3": 500,
    "L4": 500,
    "L5": 500,
}))
SMARTPHONES_REQUESTED_COUNT_BY_LEVEL = dict(globals().get("SMARTPHONES_REQUESTED_COUNT_BY_LEVEL", {
    "L1": 5,
    "L2": 5,
    "L3": 5,
    "L4": 5,
    "L5": 3,
}))

Q_SMARTPHONE = globals().get("Q_SMARTPHONE") or "Q22645"
Q_MOBILE_PHONE = globals().get("Q_MOBILE_PHONE") or "Q17517"
Q_ANDROID = globals().get("Q_ANDROID") or "Q94"
Q_IOS = globals().get("Q_IOS") or "Q48493"

SMARTPHONES_MAKER_ANCHORS = list(globals().get("SMARTPHONE_MAKER_ANCHORS", [
    ("Q312", "Apple"), ("Q27414", "Samsung"), ("Q174187", "Xiaomi"),
    ("Q175751", "Huawei"), ("Q95", "Google"), ("Q318144", "Sony"),
    ("Q3884", "Nokia"), ("Q207194", "Motorola"), ("Q215380", "OnePlus"),
    ("Q1134006", "Honor"), ("Q1078460", "Oppo"), ("Q679694", "Vivo"),
    ("Q257998", "LG"), ("Q15148", "HTC"), ("Q192608", "Lenovo"),
]))
SMARTPHONES_OS_ANCHORS = [(Q_ANDROID, "Android"), (Q_IOS, "iOS")]

_SP_ENUM_CACHE: Dict[str, List[Any]] = {}
_SP_ENUM_CURSOR: Dict[str, int] = defaultdict(int)
_SP_CANDIDATE_CACHE: Dict[Tuple[Any, ...], List[Any]] = {}
_SP_MODELS_BY_MAKER_CACHE: Dict[Tuple[str, int, int, int], List[Dict[str, Any]]] = {}

_SP_QID_RE = re.compile(r"^Q\d+$")
_SP_BAD_LABEL_EXACT = {
    "iphone", "ipad", "google pixel", "pixel", "nexus", "samsung galaxy",
    "galaxy", "samsung galaxy s", "samsung galaxy a", "sony xperia",
    "nokia lumia", "motorola moto", "xiaomi redmi", "redmi", "honor",
}
_SP_BAD_LABEL_RE = re.compile(
    r"\b(series|lineup|family|range|brand|prototype|list of|comparison of|smartphones?|mobile phones?)\b",
    re.I,
)


def _sp_norm(x: Any) -> str:
    if x is None:
        return ""
    try:
        import pandas as _pd
        if _pd.isna(x):
            return ""
    except Exception:
        pass
    s = str(x).strip()
    return "" if s.lower() == "nan" else s


def _sp_uri_to_qid(uri: Any) -> Optional[str]:
    if not uri:
        return None
    try:
        qid = uri_to_qid(str(uri))
        if qid:
            return qid
    except Exception:
        pass
    m = re.search(r"Q\d+", str(uri))
    return m.group(0) if m else None


def _sp_rows(data: Any) -> List[Dict[str, str]]:
    try:
        return rows_from_select(data)
    except Exception:
        out = []
        for b in (data or {}).get("results", {}).get("bindings", []):
            out.append({k: v.get("value") for k, v in b.items()})
        return out


def _sp_now_z() -> str:
    try:
        return utc_now_z()
    except Exception:
        return _dt.datetime.now(_dt.timezone.utc).isoformat().replace("+00:00", "Z")


def _sp_label_quality_ok(label: str) -> bool:
    lab = re.sub(r"\s+", " ", str(label or "").strip())
    if not lab or _SP_QID_RE.fullmatch(lab) or len(lab) < 3:
        return False
    low = lab.casefold()
    if low in _SP_BAD_LABEL_EXACT:
        return False
    if _SP_BAD_LABEL_RE.search(low):
        return False
    return True


def _sp_clean_constraints(c: Dict[str, Any]) -> Dict[str, Any]:
    out: Dict[str, Any] = {}
    for k, v in dict(c or {}).items():
        if k.endswith("_qid") or k.endswith("_qids") or k.endswith("_ru"):
            continue
        if v is None or v is False or v == "" or v == [] or v == {}:
            continue
        out[k] = v
    return out


def _sp_type_lines(item_var: str = "item") -> List[str]:
    # Use smartphone when Wikidata has it; include mobile-phone-with-OS fallback to
    # avoid losing many real smartphone models that are modeled as mobile phones.
    return [
        "{",
        f"  ?{item_var} wdt:P31/wdt:P279* wd:{Q_SMARTPHONE} .",
        "} UNION {",
        f"  ?{item_var} wdt:P31/wdt:P279* wd:{Q_MOBILE_PHONE} .",
        f"  ?{item_var} wdt:P306 ?_typeOs .",
        "}",
        f"?{item_var} wdt:P176 ?_typeManufacturer .",
        f"?{item_var} wikibase:sitelinks ?_sitelinks .",
        "FILTER(?_sitelinks >= 1)",
    ]


def _sp_date_lines(item_var: str = "item", date_var: str = "date", p577_var: str = "datePublished", p571_var: str = "dateInception") -> List[str]:
    return [
        f"OPTIONAL {{ ?{item_var} wdt:P577 ?{p577_var} . }}",
        f"OPTIONAL {{ ?{item_var} wdt:P571 ?{p571_var} . }}",
        f"BIND(COALESCE(?{p577_var}, ?{p571_var}) AS ?{date_var})",
        f"FILTER(BOUND(?{date_var}))",
    ]


def _sp_build_select_query(where_lines: List[str], *, answer_var: str = "item", limit: int = SMARTPHONES_WDQS_LIMIT_DEFAULT) -> str:
    where = "\n      ".join(where_lines)
    return f"""
    SELECT DISTINCT ?{answer_var} ?{answer_var}LabelEn ?{answer_var}LabelRu WHERE {{
      {where}
      ?{answer_var} rdfs:label ?{answer_var}LabelEn FILTER(LANG(?{answer_var}LabelEn) = "en") .
      OPTIONAL {{ ?{answer_var} rdfs:label ?{answer_var}LabelRu FILTER(LANG(?{answer_var}LabelRu) = "ru") . }}
    }}
    LIMIT {int(limit)}
    """.strip()


def _sp_build_ask_query(where_lines: List[str], *, answer_var: str = "item") -> str:
    where = "\n      ".join(where_lines)
    return f"""
    # WDQS-only validator. All constraints are checked directly in Wikidata.
    ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?{answer_var})
      {where}
    }}
    """.strip()


def _sp_sparql_select_safe(query: str, use_cache: bool = True) -> Any:
    try:
        import signal
        if not hasattr(signal, "SIGALRM"):
            return wd.sparql_select(query, use_cache=use_cache)
        old_handler = signal.getsignal(signal.SIGALRM)
        def _handler(signum, frame):
            raise TimeoutError(f"WDQS hard timeout after {SMARTPHONES_WDQS_HARD_TIMEOUT_SECONDS}s")
        signal.signal(signal.SIGALRM, _handler)
        signal.setitimer(signal.ITIMER_REAL, float(SMARTPHONES_WDQS_HARD_TIMEOUT_SECONDS))
        try:
            return wd.sparql_select(query, use_cache=use_cache)
        finally:
            signal.setitimer(signal.ITIMER_REAL, 0)
            signal.signal(signal.SIGALRM, old_handler)
    except TypeError:
        return wd.sparql_select(query)


def _sp_collect_gold(sparql_query: str, *, answer_var: str = "item", wdqs_limit: int = SMARTPHONES_WDQS_LIMIT_DEFAULT) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    rows = _sp_rows(_sp_sparql_select_safe(sparql_query, use_cache=True))
    seen = set()
    items: List[Dict[str, Any]] = []
    dropped_no_qid = dropped_no_en = dropped_bad_label = 0
    ru_count = en_fallback = 0

    for row in rows:
        qid = _sp_uri_to_qid(row.get(answer_var))
        if not qid:
            dropped_no_qid += 1
            continue
        if qid in seen:
            continue
        label_en = _sp_norm(row.get(f"{answer_var}LabelEn"))
        label_ru = _sp_norm(row.get(f"{answer_var}LabelRu"))
        if not label_en or _SP_QID_RE.fullmatch(label_en):
            dropped_no_en += 1
            continue
        if not _sp_label_quality_ok(label_en) or (label_ru and not _sp_label_quality_ok(label_ru)):
            dropped_bad_label += 1
            continue
        if label_ru and not _SP_QID_RE.fullmatch(label_ru):
            ru_count += 1
        else:
            label_ru = label_en
            en_fallback += 1
        seen.add(qid)
        items.append({"qid": qid, "label_ru": label_ru, "label_en": label_en})

    meta = {
        "source": "wikidata_sparql",
        "wdqs_candidate_limit": int(wdqs_limit),
        "rows_returned_by_wdqs": len(rows),
        "gold_returned_before_limits": len(items),
        "dropped_no_qid_count": dropped_no_qid,
        "dropped_no_en_label_count": dropped_no_en,
        "dropped_bad_label_count": dropped_bad_label,
        "label_sources": {"ru_label": ru_count, "en_fallback_for_ru": en_fallback},
        "gold_may_be_incomplete_due_to_wdqs_limit": len(rows) >= int(wdqs_limit),
    }
    return items, meta


def _sp_duplicate_labels(labels: List[str]) -> Dict[str, int]:
    norm = [re.sub(r"\s+", " ", str(x or "").strip().casefold()) for x in labels if str(x or "").strip()]
    c = Counter(norm)
    return {k: v for k, v in c.items() if v > 1}


def _sp_gold_quality_reject_reason(gold_items: List[Dict[str, Any]]) -> Optional[str]:
    qids = [x.get("qid") for x in gold_items]
    if len(qids) != len(set(qids)):
        return "duplicate_gold_qids"
    dup_en = _sp_duplicate_labels([x.get("label_en", "") for x in gold_items])
    if dup_en:
        return f"duplicate_en_gold_labels:{sorted(dup_en)[:5]}"
    dup_ru = _sp_duplicate_labels([x.get("label_ru", "") for x in gold_items])
    if dup_ru:
        return f"duplicate_ru_gold_labels:{sorted(dup_ru)[:5]}"
    return None


def _sp_finalize_wdqs_example(
    *,
    idx: int,
    complexity: str,
    template_id: str,
    template_family: str,
    query_text_ru: str,
    query_text_en: str,
    constraints: Dict[str, Any],
    where_lines: List[str],
    requested_count: int,
    answer_var: str = "item",
    bridge_meta: Optional[Dict[str, Any]] = None,
    wdqs_limit: int = SMARTPHONES_WDQS_LIMIT_DEFAULT,
) -> Optional[BenchmarkExample]:
    sparql_query = _sp_build_select_query(where_lines, answer_var=answer_var, limit=wdqs_limit)
    try:
        gold_items, meta = _sp_collect_gold(sparql_query, answer_var=answer_var, wdqs_limit=wdqs_limit)
    except Exception as e:
        if globals().get("DEBUG_GENERATOR_ERRORS", False):
            print(f"[smartphones:WDQS skip] {template_id}: {e}")
        return None

    min_gold = max(int(requested_count), SMARTPHONES_MIN_GOLD_BY_LEVEL.get(complexity, int(requested_count)))
    if complexity != "L5":
        min_gold = max(min_gold, int(requested_count) + 1)
    if len(gold_items) < min_gold:
        return None

    quality_reason = _sp_gold_quality_reject_reason(gold_items)
    if quality_reason:
        if globals().get("DEBUG_GENERATOR_ERRORS", False):
            print(f"[smartphones:quality skip] {template_id}: {quality_reason}")
        return None

    max_gold = int(SMARTPHONES_MAX_GOLD_BY_LEVEL.get(complexity, 500))
    gold_truncated_by_local_limit = len(gold_items) > max_gold
    wdqs_limit_hit = bool(meta.get("gold_may_be_incomplete_due_to_wdqs_limit"))
    if SMARTPHONES_STRICT_FULL_GOLD and (gold_truncated_by_local_limit or wdqs_limit_hit):
        return None

    limited_gold = gold_items[:max_gold]
    cleaned_constraints = _sp_clean_constraints(constraints)
    meta.update({
        "constraints_are_wdqs_only": True,
        "gold_limit": max_gold,
        "gold_returned": len(limited_gold),
        "gold_total_before_limit": len(gold_items),
        "gold_truncated_by_local_limit": gold_truncated_by_local_limit,
        "template_id": template_id,
        "template_family": template_family,
    })
    if bridge_meta:
        meta["bridge_meta"] = bridge_meta

    local_validator = {
        "type": "none_wdqs_only",
        "source": "Wikidata Query Service",
        "applies_after": "ask_validator_sparql",
        "filters": cleaned_constraints,
        "label_matching_used": False,
        "note": "All constraints for this smartphones task are represented in the WDQS ASK validator; no external local validator is required.",
    }

    return BenchmarkExample(
        id=f"smartphones_{complexity.lower()}_{idx:05d}",
        domain=DOMAIN,
        complexity=complexity,
        query_text_ru=query_text_ru,
        query_text_en=query_text_en,
        constraints=cleaned_constraints,
        requested_count=int(requested_count),
        gold_answer_qids=[x["qid"] for x in limited_gold],
        gold_answer_labels_ru=[x["label_ru"] for x in limited_gold],
        gold_answer_labels_en=[x["label_en"] for x in limited_gold],
        sparql_query=sparql_query,
        created_at=_sp_now_z(),
        is_advanced=complexity in {"L3", "L4", "L5"},
        template_id=template_id,
        template_family=template_family,
        gold_truncated=bool(gold_truncated_by_local_limit or wdqs_limit_hit),
        ask_validator_sparql=_sp_build_ask_query(where_lines, answer_var=answer_var),
        local_validator=local_validator,
        gold_collection_meta=meta,
        gold_answer_imdb_ids=[],
        gold_answer_imdb_titles=[],
    )


def _sp_next_candidate(key: str, builder, rng: random.Random):
    if key not in _SP_ENUM_CACHE:
        items = list(builder() or [])
        rng.shuffle(items)
        _SP_ENUM_CACHE[key] = items
        _SP_ENUM_CURSOR[key] = 0
    items = _SP_ENUM_CACHE.get(key, [])
    if not items:
        return None
    cur = _SP_ENUM_CURSOR.get(key, 0)
    if cur >= len(items):
        if len(items) > 1:
            rng.shuffle(items)
        cur = 0
    _SP_ENUM_CURSOR[key] = cur + 1
    return items[cur]


def _sp_fallback_makers() -> List[Tuple[str, str, str, int]]:
    return [(qid, label, label, 999) for qid, label in SMARTPHONES_MAKER_ANCHORS]


def find_sp_maker_candidates(min_models: int = 6, limit: int = 100) -> List[Tuple[str, str, str, int]]:
    key = ("maker", int(min_models), int(limit))
    if key in _SP_CANDIDATE_CACHE:
        return _SP_CANDIDATE_CACHE[key]
    where_lines = [*_sp_type_lines("item"), "?item wdt:P176 ?maker ."]
    where = "\n      ".join(where_lines)
    sparql = f"""
    SELECT ?maker ?makerLabelEn ?makerLabelRu (COUNT(DISTINCT ?item) AS ?cnt) WHERE {{
      {where}
      ?maker rdfs:label ?makerLabelEn FILTER(LANG(?makerLabelEn) = "en") .
      OPTIONAL {{ ?maker rdfs:label ?makerLabelRu FILTER(LANG(?makerLabelRu) = "ru") . }}
    }}
    GROUP BY ?maker ?makerLabelEn ?makerLabelRu
    HAVING(COUNT(DISTINCT ?item) >= {int(min_models)})
    ORDER BY DESC(COUNT(DISTINCT ?item))
    LIMIT {int(limit)}
    """.strip()
    out: List[Tuple[str, str, str, int]] = []
    try:
        rows = _sp_rows(_sp_sparql_select_safe(sparql, use_cache=True))
        for r in rows:
            qid = _sp_uri_to_qid(r.get("maker"))
            en = _sp_norm(r.get("makerLabelEn"))
            ru = _sp_norm(r.get("makerLabelRu")) or en
            try:
                cnt = int(float(r.get("cnt", 0)))
            except Exception:
                cnt = 0
            if qid and en and cnt >= int(min_models):
                out.append((qid, ru, en, cnt))
    except Exception:
        out = []
    if not out:
        out = _sp_fallback_makers()
    _SP_CANDIDATE_CACHE[key] = out
    return out


def find_sp_maker_os_candidates(min_models: int = 5, limit: int = 160) -> List[Tuple[str, str, str, str, str, str, int]]:
    key = ("maker_os", int(min_models), int(limit))
    if key in _SP_CANDIDATE_CACHE:
        return _SP_CANDIDATE_CACHE[key]
    where_lines = [*_sp_type_lines("item"), "?item wdt:P176 ?maker .", "?item wdt:P306 ?os ."]
    where = "\n      ".join(where_lines)
    sparql = f"""
    SELECT ?maker ?makerLabelEn ?makerLabelRu ?os ?osLabelEn ?osLabelRu (COUNT(DISTINCT ?item) AS ?cnt) WHERE {{
      {where}
      ?maker rdfs:label ?makerLabelEn FILTER(LANG(?makerLabelEn) = "en") .
      OPTIONAL {{ ?maker rdfs:label ?makerLabelRu FILTER(LANG(?makerLabelRu) = "ru") . }}
      ?os rdfs:label ?osLabelEn FILTER(LANG(?osLabelEn) = "en") .
      OPTIONAL {{ ?os rdfs:label ?osLabelRu FILTER(LANG(?osLabelRu) = "ru") . }}
    }}
    GROUP BY ?maker ?makerLabelEn ?makerLabelRu ?os ?osLabelEn ?osLabelRu
    HAVING(COUNT(DISTINCT ?item) >= {int(min_models)})
    ORDER BY DESC(COUNT(DISTINCT ?item))
    LIMIT {int(limit)}
    """.strip()
    out: List[Tuple[str, str, str, str, str, str, int]] = []
    try:
        rows = _sp_rows(_sp_sparql_select_safe(sparql, use_cache=True))
        for r in rows:
            maker_qid = _sp_uri_to_qid(r.get("maker"))
            os_qid = _sp_uri_to_qid(r.get("os"))
            maker_en = _sp_norm(r.get("makerLabelEn"))
            maker_ru = _sp_norm(r.get("makerLabelRu")) or maker_en
            os_en = _sp_norm(r.get("osLabelEn"))
            os_ru = _sp_norm(r.get("osLabelRu")) or os_en
            try:
                cnt = int(float(r.get("cnt", 0)))
            except Exception:
                cnt = 0
            if maker_qid and os_qid and maker_en and os_en and cnt >= int(min_models):
                out.append((maker_qid, maker_ru, maker_en, os_qid, os_ru, os_en, cnt))
    except Exception:
        out = []
    if not out:
        for mq, mr, me, _ in _sp_fallback_makers():
            for oq, oe in SMARTPHONES_OS_ANCHORS:
                out.append((mq, mr, me, oq, oe, oe, max(min_models, 5)))
    _SP_CANDIDATE_CACHE[key] = out
    return out


def _sp_parse_year(date_value: Any) -> Optional[int]:
    m = re.match(r"^(\d{4})", _sp_norm(date_value))
    return int(m.group(1)) if m else None


def fetch_sp_models_for_maker(maker_qid: str, min_year: int = 2007, max_year: int = 2026, limit: int = 450) -> List[Dict[str, Any]]:
    key = (str(maker_qid), int(min_year), int(max_year), int(limit))
    if key in _SP_MODELS_BY_MAKER_CACHE:
        return _SP_MODELS_BY_MAKER_CACHE[key]
    where_lines = [
        *_sp_type_lines("item"),
        f"?item wdt:P176 wd:{maker_qid} .",
        *_sp_date_lines("item", "date", "datePublished", "dateInception"),
        f"FILTER(YEAR(?date) >= {int(min_year)} && YEAR(?date) <= {int(max_year)})",
        "OPTIONAL { ?item wdt:P306 ?os . ?os rdfs:label ?osLabelEn FILTER(LANG(?osLabelEn) = \"en\") . OPTIONAL { ?os rdfs:label ?osLabelRu FILTER(LANG(?osLabelRu) = \"ru\") . } }",
    ]
    where = "\n      ".join(where_lines)
    sparql = f"""
    SELECT DISTINCT ?item ?itemLabelEn ?itemLabelRu ?date ?os ?osLabelEn ?osLabelRu WHERE {{
      {where}
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
      OPTIONAL {{ ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") . }}
    }}
    ORDER BY ASC(?date) ?itemLabelEn
    LIMIT {int(limit)}
    """.strip()
    acc: Dict[str, Dict[str, Any]] = {}
    try:
        rows = _sp_rows(_sp_sparql_select_safe(sparql, use_cache=True))
    except Exception:
        rows = []
    for r in rows:
        qid = _sp_uri_to_qid(r.get("item"))
        en = _sp_norm(r.get("itemLabelEn"))
        ru = _sp_norm(r.get("itemLabelRu")) or en
        yy = _sp_parse_year(r.get("date"))
        if not qid or not en or yy is None or not _sp_label_quality_ok(en) or not _sp_label_quality_ok(ru):
            continue
        rec = acc.setdefault(qid, {
            "qid": qid,
            "label_ru": ru,
            "label_en": en,
            "date": _sp_norm(r.get("date")),
            "year": int(yy),
            "os_qids": set(),
            "os_labels_ru": {},
            "os_labels_en": {},
        })
        os_qid = _sp_uri_to_qid(r.get("os"))
        os_en = _sp_norm(r.get("osLabelEn"))
        os_ru = _sp_norm(r.get("osLabelRu")) or os_en
        if os_qid and os_en:
            rec["os_qids"].add(os_qid)
            rec["os_labels_en"][os_qid] = os_en
            rec["os_labels_ru"][os_qid] = os_ru
    out = sorted(acc.values(), key=lambda x: (x["year"], x["label_en"]))
    _SP_MODELS_BY_MAKER_CACHE[key] = out
    return out


def find_sp_maker_year_window_candidates(min_models: int = 5, maker_limit: int = 28, min_span_years: int = 2, max_span_years: int = 4, max_candidates_per_maker: int = 10) -> List[Tuple[str, str, str, int, int, int]]:
    key = ("maker_year_window", min_models, maker_limit, min_span_years, max_span_years, max_candidates_per_maker)
    if key in _SP_CANDIDATE_CACHE:
        return _SP_CANDIDATE_CACHE[key]
    out: List[Tuple[str, str, str, int, int, int]] = []
    for maker_qid, maker_ru, maker_en, _ in find_sp_maker_candidates(min_models=max(min_models + 1, 6), limit=maker_limit):
        models = fetch_sp_models_for_maker(maker_qid)
        year_counts: Dict[int, int] = defaultdict(int)
        for m in models:
            year_counts[int(m["year"])] += 1
        years = sorted(year_counts)
        local = []
        for i, y1 in enumerate(years):
            total = 0
            for y2 in years[i:]:
                total += year_counts[y2]
                span = y2 - y1 + 1
                if span < min_span_years:
                    continue
                if span > max_span_years:
                    break
                if total >= min_models:
                    local.append((maker_qid, maker_ru, maker_en, y1, y2, total))
        local.sort(key=lambda x: (abs(x[5] - 8), abs((x[4]-x[3]+1)-3), x[3], x[4]))
        out.extend(local[:max_candidates_per_maker])
    _SP_CANDIDATE_CACHE[key] = out
    return out


def find_sp_maker_os_year_window_candidates(min_models: int = 5, maker_limit: int = 28, min_span_years: int = 2, max_span_years: int = 5, max_candidates_per_maker: int = 12) -> List[Tuple[str, str, str, str, str, str, int, int, int]]:
    key = ("maker_os_year_window", min_models, maker_limit, min_span_years, max_span_years, max_candidates_per_maker)
    if key in _SP_CANDIDATE_CACHE:
        return _SP_CANDIDATE_CACHE[key]
    out: List[Tuple[str, str, str, str, str, str, int, int, int]] = []
    for maker_qid, maker_ru, maker_en, _ in find_sp_maker_candidates(min_models=max(min_models + 1, 6), limit=maker_limit):
        models = fetch_sp_models_for_maker(maker_qid)
        os_year_counts: Dict[str, Dict[int, int]] = defaultdict(lambda: defaultdict(int))
        os_labels_ru: Dict[str, str] = {}
        os_labels_en: Dict[str, str] = {}
        for m in models:
            for os_qid in m.get("os_qids", set()):
                os_year_counts[os_qid][int(m["year"])] += 1
                os_labels_ru[os_qid] = m.get("os_labels_ru", {}).get(os_qid) or os_qid
                os_labels_en[os_qid] = m.get("os_labels_en", {}).get(os_qid) or os_qid
        local = []
        for os_qid, year_counts in os_year_counts.items():
            years = sorted(year_counts)
            for i, y1 in enumerate(years):
                total = 0
                for y2 in years[i:]:
                    total += year_counts[y2]
                    span = y2 - y1 + 1
                    if span < min_span_years:
                        continue
                    if span > max_span_years:
                        break
                    if total >= min_models:
                        local.append((maker_qid, maker_ru, maker_en, os_qid, os_labels_ru.get(os_qid, os_qid), os_labels_en.get(os_qid, os_qid), y1, y2, total))
        local.sort(key=lambda x: (abs(x[8] - 7), abs((x[7]-x[6]+1)-3), x[6], x[7]))
        out.extend(local[:max_candidates_per_maker])
    _SP_CANDIDATE_CACHE[key] = out
    return out


def find_sp_reference_candidates(min_models: int = 5, maker_limit: int = 24, max_candidates_per_maker: int = 12) -> List[Tuple[str, str, str, str, str, str, int, str, int]]:
    key = ("reference", min_models, maker_limit, max_candidates_per_maker)
    if key in _SP_CANDIDATE_CACHE:
        return _SP_CANDIDATE_CACHE[key]
    out = []
    for maker_qid, maker_ru, maker_en, _ in find_sp_maker_candidates(min_models=max(min_models + 2, 7), limit=maker_limit):
        models = fetch_sp_models_for_maker(maker_qid)
        n = len(models)
        if n < min_models + 2:
            continue
        local = []
        for i, m in enumerate(models):
            cnt_before = i
            cnt_after = n - i - 1
            if cnt_after >= min_models and cnt_before >= 1:
                local.append((maker_qid, maker_ru, maker_en, m["qid"], m["label_ru"], m["label_en"], int(m["year"]), "after", cnt_after))
            if cnt_before >= min_models and cnt_after >= 1:
                local.append((maker_qid, maker_ru, maker_en, m["qid"], m["label_ru"], m["label_en"], int(m["year"]), "before", cnt_before))
        local.sort(key=lambda x: (abs(x[8] - 8), abs(x[6] - 2018), 0 if x[7] == "after" else 1))
        out.extend(local[:max_candidates_per_maker])
    _SP_CANDIDATE_CACHE[key] = out
    return out


def find_sp_hidden_maker_year_candidates(min_models: int = 5, maker_limit: int = 24, max_candidates_per_maker: int = 10) -> List[Tuple[str, str, str, str, str, str, int, int, int]]:
    key = ("hidden_maker_year", min_models, maker_limit, max_candidates_per_maker)
    if key in _SP_CANDIDATE_CACHE:
        return _SP_CANDIDATE_CACHE[key]
    out = []
    for maker_qid, maker_ru, maker_en, y1, y2, cnt in find_sp_maker_year_window_candidates(min_models=min_models, maker_limit=maker_limit, max_candidates_per_maker=max_candidates_per_maker):
        models = fetch_sp_models_for_maker(maker_qid)
        seed = next((m for m in models if not (y1 <= int(m["year"]) <= y2)), None) or (models[0] if models else None)
        if seed:
            out.append((maker_qid, maker_ru, maker_en, seed["qid"], seed["label_ru"], seed["label_en"], y1, y2, cnt))
    _SP_CANDIDATE_CACHE[key] = out
    return out


def find_sp_os_seed_models(limit: int = 80) -> List[Tuple[str, str, str, str, str]]:
    key = ("os_seed_models", limit)
    if key in _SP_CANDIDATE_CACHE:
        return _SP_CANDIDATE_CACHE[key]
    where_lines = [*_sp_type_lines("seed"), "?seed wdt:P306 ?os .", *_sp_date_lines("seed", "seedDate", "sd1", "sd2")]
    where = "\n      ".join(where_lines)
    sparql = f"""
    SELECT DISTINCT ?seed ?seedLabelEn ?seedLabelRu ?os ?osLabelEn ?osLabelRu WHERE {{
      {where}
      ?seed rdfs:label ?seedLabelEn FILTER(LANG(?seedLabelEn) = "en") .
      OPTIONAL {{ ?seed rdfs:label ?seedLabelRu FILTER(LANG(?seedLabelRu) = "ru") . }}
      ?os rdfs:label ?osLabelEn FILTER(LANG(?osLabelEn) = "en") .
      OPTIONAL {{ ?os rdfs:label ?osLabelRu FILTER(LANG(?osLabelRu) = "ru") . }}
    }}
    ORDER BY DESC(?seedLabelEn)
    LIMIT {int(limit)}
    """.strip()
    out = []
    try:
        rows = _sp_rows(_sp_sparql_select_safe(sparql, use_cache=True))
        for r in rows:
            seed_qid = _sp_uri_to_qid(r.get("seed"))
            os_qid = _sp_uri_to_qid(r.get("os"))
            seed_en = _sp_norm(r.get("seedLabelEn"))
            seed_ru = _sp_norm(r.get("seedLabelRu")) or seed_en
            os_en = _sp_norm(r.get("osLabelEn"))
            if seed_qid and os_qid and seed_en and os_en and _sp_label_quality_ok(seed_en):
                out.append((seed_qid, seed_ru, seed_en, os_qid, os_en))
    except Exception:
        out = []
    _SP_CANDIDATE_CACHE[key] = out
    return out


def find_sp_hidden_os_maker_window_candidates(min_models: int = 5, maker_limit: int = 24, max_candidates: int = 120) -> List[Tuple[str, str, str, str, str, str, str, str, str, int, int, int]]:
    key = ("hidden_os_maker_window", min_models, maker_limit, max_candidates)
    if key in _SP_CANDIDATE_CACHE:
        return _SP_CANDIDATE_CACHE[key]
    seeds = find_sp_os_seed_models(limit=120)
    seeds_by_os: Dict[str, List[Tuple[str, str, str, str, str]]] = defaultdict(list)
    for s in seeds:
        seeds_by_os[s[3]].append(s)
    out = []
    for maker_qid, maker_ru, maker_en, os_qid, os_ru, os_en, y1, y2, cnt in find_sp_maker_os_year_window_candidates(min_models=min_models, maker_limit=maker_limit):
        seed = next((s for s in seeds_by_os.get(os_qid, []) if True), None)
        if seed:
            seed_qid, seed_ru, seed_en, _, _ = seed
            out.append((maker_qid, maker_ru, maker_en, seed_qid, seed_ru, seed_en, os_qid, os_ru, os_en, y1, y2, cnt))
    out.sort(key=lambda x: (abs(x[11] - 7), x[9], x[10]))
    _SP_CANDIDATE_CACHE[key] = out[:max_candidates]
    return _SP_CANDIDATE_CACHE[key]


def find_sp_l5_between_hidden_candidates(min_models_between: int = 3, maker_limit: int = 24, max_candidates_per_maker: int = 16) -> List[Tuple[Any, ...]]:
    key = ("l5_between_hidden", min_models_between, maker_limit, max_candidates_per_maker)
    if key in _SP_CANDIDATE_CACHE:
        return _SP_CANDIDATE_CACHE[key]
    out = []
    for maker_qid, maker_ru, maker_en, _ in find_sp_maker_candidates(min_models=max(min_models_between + 4, 8), limit=maker_limit):
        models = fetch_sp_models_for_maker(maker_qid)
        n = len(models)
        if n < min_models_between + 3:
            continue
        local = []
        for i in range(n):
            left = models[i]
            for j in range(i + min_models_between + 1, n):
                right = models[j]
                if int(right["year"]) <= int(left["year"]):
                    continue
                between = models[i+1:j]
                os_counts: Dict[str, int] = defaultdict(int)
                for m in between:
                    for os_qid in m.get("os_qids", set()):
                        os_counts[os_qid] += 1
                for os_qid, cnt in os_counts.items():
                    if cnt < min_models_between:
                        continue
                    seed_maker = next((m for m in models if m["qid"] not in {left["qid"], right["qid"]}), None)
                    seed_os = next((m for m in models if os_qid in m.get("os_qids", set()) and m["qid"] not in {left["qid"], right["qid"]}), None)
                    if not seed_maker or not seed_os:
                        continue
                    os_ru = seed_os.get("os_labels_ru", {}).get(os_qid, os_qid)
                    os_en = seed_os.get("os_labels_en", {}).get(os_qid, os_qid)
                    local.append((
                        maker_qid, maker_ru, maker_en,
                        seed_maker["qid"], seed_maker["label_ru"], seed_maker["label_en"],
                        seed_os["qid"], seed_os["label_ru"], seed_os["label_en"], os_qid, os_ru, os_en,
                        left["qid"], left["label_ru"], left["label_en"], int(left["year"]),
                        right["qid"], right["label_ru"], right["label_en"], int(right["year"]),
                        cnt,
                    ))
        local.sort(key=lambda x: (abs(x[20] - 4), abs((x[19] - x[15]) - 4), x[15], x[19]))
        out.extend(local[:max_candidates_per_maker])
    _SP_CANDIDATE_CACHE[key] = out
    return out


def _sp_where_maker(maker_qid: str) -> List[str]:
    return [*_sp_type_lines("item"), f"?item wdt:P176 wd:{maker_qid} ."]


def _sp_where_maker_os_year(maker_qid: str, os_qid: str, y1: int, y2: int) -> List[str]:
    return [
        *_sp_type_lines("item"),
        f"?item wdt:P176 wd:{maker_qid} .",
        f"?item wdt:P306 wd:{os_qid} .",
        *_sp_date_lines("item", "date", "datePublished", "dateInception"),
        f"FILTER(YEAR(?date) >= {int(y1)} && YEAR(?date) <= {int(y2)})",
    ]


def _sp_where_maker_year(maker_qid: str, y1: int, y2: int) -> List[str]:
    return [
        *_sp_type_lines("item"),
        f"?item wdt:P176 wd:{maker_qid} .",
        *_sp_date_lines("item", "date", "datePublished", "dateInception"),
        f"FILTER(YEAR(?date) >= {int(y1)} && YEAR(?date) <= {int(y2)})",
    ]


def _sp_where_reference(maker_qid: str, ref_qid: str, relation: str) -> List[str]:
    lines = [
        *_sp_type_lines("item"),
        f"?item wdt:P176 wd:{maker_qid} .",
        *_sp_date_lines("item", "date", "datePublished", "dateInception"),
        f"VALUES ?referenceModel {{ wd:{ref_qid} }}",
        *_sp_date_lines("referenceModel", "referenceDate", "refPublished", "refInception"),
        "FILTER(?item != ?referenceModel)",
    ]
    if relation == "after":
        lines.append("FILTER(?date > ?referenceDate)")
    else:
        lines.append("FILTER(?date < ?referenceDate)")
    return lines


def _sp_where_hidden_maker_year(seed_qid: str, y1: int, y2: int) -> List[str]:
    return [
        *_sp_type_lines("item"),
        f"VALUES ?seedMakerModel {{ wd:{seed_qid} }}",
        "?seedMakerModel wdt:P176 ?bridgeManufacturer .",
        "?item wdt:P176 ?bridgeManufacturer .",
        "FILTER(?item != ?seedMakerModel)",
        *_sp_date_lines("item", "date", "datePublished", "dateInception"),
        f"FILTER(YEAR(?date) >= {int(y1)} && YEAR(?date) <= {int(y2)})",
    ]


def _sp_where_hidden_os_maker_window(maker_qid: str, seed_os_qid: str, y1: int, y2: int) -> List[str]:
    return [
        *_sp_type_lines("item"),
        f"?item wdt:P176 wd:{maker_qid} .",
        f"VALUES ?seedOsModel {{ wd:{seed_os_qid} }}",
        "?seedOsModel wdt:P306 ?bridgeOperatingSystem .",
        "?item wdt:P306 ?bridgeOperatingSystem .",
        "FILTER(?item != ?seedOsModel)",
        *_sp_date_lines("item", "date", "datePublished", "dateInception"),
        f"FILTER(YEAR(?date) >= {int(y1)} && YEAR(?date) <= {int(y2)})",
    ]


def _sp_where_l5_between_hidden(seed_maker_qid: str, seed_os_qid: str, left_qid: str, right_qid: str) -> List[str]:
    return [
        *_sp_type_lines("item"),
        f"VALUES ?seedMakerModel {{ wd:{seed_maker_qid} }}",
        "?seedMakerModel wdt:P176 ?bridgeManufacturer .",
        "?item wdt:P176 ?bridgeManufacturer .",
        f"VALUES ?seedOsModel {{ wd:{seed_os_qid} }}",
        "?seedOsModel wdt:P306 ?bridgeOperatingSystem .",
        "?item wdt:P306 ?bridgeOperatingSystem .",
        f"VALUES ?leftReferenceModel {{ wd:{left_qid} }}",
        *_sp_date_lines("leftReferenceModel", "leftReferenceDate", "leftPublished", "leftInception"),
        f"VALUES ?rightReferenceModel {{ wd:{right_qid} }}",
        *_sp_date_lines("rightReferenceModel", "rightReferenceDate", "rightPublished", "rightInception"),
        *_sp_date_lines("item", "date", "datePublished", "dateInception"),
        "FILTER(?leftReferenceDate < ?rightReferenceDate)",
        "FILTER(?date > ?leftReferenceDate && ?date < ?rightReferenceDate)",
        "FILTER(?item != ?seedMakerModel && ?item != ?seedOsModel && ?item != ?leftReferenceModel && ?item != ?rightReferenceModel)",
    ]


def _sp_where_l5_between_hidden_maker_direct_os(seed_maker_qid: str, os_qid: str, left_qid: str, right_qid: str) -> List[str]:
    return [
        *_sp_type_lines("item"),
        f"VALUES ?seedMakerModel {{ wd:{seed_maker_qid} }}",
        "?seedMakerModel wdt:P176 ?bridgeManufacturer .",
        "?item wdt:P176 ?bridgeManufacturer .",
        f"?item wdt:P306 wd:{os_qid} .",
        f"VALUES ?leftReferenceModel {{ wd:{left_qid} }}",
        *_sp_date_lines("leftReferenceModel", "leftReferenceDate", "leftPublished", "leftInception"),
        f"VALUES ?rightReferenceModel {{ wd:{right_qid} }}",
        *_sp_date_lines("rightReferenceModel", "rightReferenceDate", "rightPublished", "rightInception"),
        *_sp_date_lines("item", "date", "datePublished", "dateInception"),
        "FILTER(?leftReferenceDate < ?rightReferenceDate)",
        "FILTER(?date > ?leftReferenceDate && ?date < ?rightReferenceDate)",
        "FILTER(?item != ?seedMakerModel && ?item != ?leftReferenceModel && ?item != ?rightReferenceModel)",
    ]


def _sp_make_from_template(complexity: str, idx: int, rng: random.Random, template: str) -> Optional[BenchmarkExample]:
    requested = int(SMARTPHONES_REQUESTED_COUNT_BY_LEVEL.get(complexity, 5))

    if template == "l1_maker":
        cand = _sp_next_candidate("l1_maker", lambda: find_sp_maker_candidates(min_models=requested + 1, limit=120), rng)
        if not cand: return None
        maker_qid, maker_ru, maker_en, _ = cand
        return _sp_finalize_wdqs_example(
            idx=idx, complexity=complexity,
            template_id="smartphones_l1_manufacturer", template_family="simple_manufacturer",
            query_text_ru=f"Назови {requested} моделей смартфонов производителя «{maker_ru}».",
            query_text_en=f"Name {requested} smartphone models manufactured by {maker_en}.",
            constraints={"manufacturer": maker_en},
            where_lines=_sp_where_maker(maker_qid), requested_count=requested,
        )

    if template == "l2_maker_os":
        cand = _sp_next_candidate("l2_maker_os", lambda: find_sp_maker_os_candidates(min_models=requested + 1, limit=180), rng)
        if not cand: return None
        maker_qid, maker_ru, maker_en, os_qid, os_ru, os_en, _ = cand
        return _sp_finalize_wdqs_example(
            idx=idx, complexity=complexity,
            template_id="smartphones_l2_manufacturer_os", template_family="manufacturer_os",
            query_text_ru=f"Назови {requested} смартфонов производителя «{maker_ru}», работающих на {os_ru}.",
            query_text_en=f"Name {requested} smartphones manufactured by {maker_en} that run on {os_en}.",
            constraints={"manufacturer": maker_en, "operating_system": os_en},
            where_lines=[*_sp_type_lines("item"), f"?item wdt:P176 wd:{maker_qid} .", f"?item wdt:P306 wd:{os_qid} ."],
            requested_count=requested,
        )

    if template == "l2_maker_year":
        cand = _sp_next_candidate("l2_maker_year", lambda: find_sp_maker_year_window_candidates(min_models=requested + 1), rng)
        if not cand: return None
        maker_qid, maker_ru, maker_en, y1, y2, _ = cand
        return _sp_finalize_wdqs_example(
            idx=idx, complexity=complexity,
            template_id="smartphones_l2_manufacturer_year_window", template_family="manufacturer_year_window",
            query_text_ru=f"Назови {requested} смартфонов производителя «{maker_ru}», выпущенных в {y1}–{y2} годах.",
            query_text_en=f"Name {requested} smartphones manufactured by {maker_en} that were released between {y1} and {y2}.",
            constraints={"manufacturer": maker_en, "year_from": y1, "year_to": y2},
            where_lines=_sp_where_maker_year(maker_qid, y1, y2), requested_count=requested,
        )

    if template == "l3_maker_os_year":
        cand = _sp_next_candidate("l3_maker_os_year", lambda: find_sp_maker_os_year_window_candidates(min_models=requested + 1), rng)
        if not cand: return None
        maker_qid, maker_ru, maker_en, os_qid, os_ru, os_en, y1, y2, _ = cand
        return _sp_finalize_wdqs_example(
            idx=idx, complexity=complexity,
            template_id="smartphones_l3_manufacturer_os_year", template_family="manufacturer_os_year_window",
            query_text_ru=f"Назови {requested} смартфонов производителя «{maker_ru}», работающих на {os_ru}, выпущенных в {y1}–{y2} годах.",
            query_text_en=f"Name {requested} smartphones manufactured by {maker_en}, running on {os_en}, and released between {y1} and {y2}.",
            constraints={"manufacturer": maker_en, "operating_system": os_en, "year_from": y1, "year_to": y2},
            where_lines=_sp_where_maker_os_year(maker_qid, os_qid, y1, y2), requested_count=requested,
        )

    if template == "l3_hidden_maker_year":
        cand = _sp_next_candidate("l3_hidden_maker_year", lambda: find_sp_hidden_maker_year_candidates(min_models=requested + 1), rng)
        if not cand: return None
        maker_qid, maker_ru, maker_en, seed_qid, seed_ru, seed_en, y1, y2, _ = cand
        return _sp_finalize_wdqs_example(
            idx=idx, complexity=complexity,
            template_id="smartphones_l3_same_manufacturer_as_model_year", template_family="same_manufacturer_year",
            query_text_ru=f"Назови {requested} смартфонов, выпущенных тем же производителем, что и «{seed_ru}», в {y1}–{y2} годах.",
            query_text_en=f"Name {requested} smartphones made by the same manufacturer as {seed_en} and released between {y1} and {y2}.",
            constraints={"manufacturer_from_model": seed_en, "year_from": y1, "year_to": y2},
            where_lines=_sp_where_hidden_maker_year(seed_qid, y1, y2), requested_count=requested,
            bridge_meta={"bridge": "manufacturer", "constraint_key": "manufacturer_from_model", "semantics": "shared_or_related_value", "intermediate_value_hidden_in_query": True, "wikidata_property": "P176", "seed_label_en": seed_en, "seed_label_ru": seed_ru},
        )

    if template == "l4_reference":
        cand = _sp_next_candidate("l4_reference", lambda: find_sp_reference_candidates(min_models=requested + 1), rng)
        if not cand: return None
        maker_qid, maker_ru, maker_en, ref_qid, ref_ru, ref_en, ref_year, relation, _ = cand
        if relation == "after":
            ru = f"Назови {requested} смартфонов производителя «{maker_ru}», выпущенных позже модели «{ref_ru}» ({ref_year})."
            en = f"Name {requested} smartphones manufactured by {maker_en} that were released after {ref_en} ({ref_year})."
            constraints = {"manufacturer": maker_en, "released_after_model": ref_en, "reference_model_year": ref_year}
        else:
            ru = f"Назови {requested} смартфонов производителя «{maker_ru}», выпущенных раньше модели «{ref_ru}» ({ref_year})."
            en = f"Name {requested} smartphones manufactured by {maker_en} that were released before {ref_en} ({ref_year})."
            constraints = {"manufacturer": maker_en, "released_before_model": ref_en, "reference_model_year": ref_year}
        return _sp_finalize_wdqs_example(
            idx=idx, complexity=complexity,
            template_id=f"smartphones_l4_manufacturer_{relation}_reference_model", template_family="manufacturer_reference_model",
            query_text_ru=ru, query_text_en=en, constraints=constraints,
            where_lines=_sp_where_reference(maker_qid, ref_qid, relation), requested_count=requested,
        )

    if template == "l4_hidden_os_maker_window":
        cand = _sp_next_candidate("l4_hidden_os_maker_window", lambda: find_sp_hidden_os_maker_window_candidates(min_models=requested + 1), rng)
        if not cand: return None
        maker_qid, maker_ru, maker_en, seed_qid, seed_ru, seed_en, os_qid, os_ru, os_en, y1, y2, _ = cand
        return _sp_finalize_wdqs_example(
            idx=idx, complexity=complexity,
            template_id="smartphones_l4_manufacturer_same_os_as_model_year", template_family="same_operating_system_manufacturer_year",
            query_text_ru=f"Назови {requested} смартфонов производителя «{maker_ru}», работающих хотя бы на одной из тех же операционных систем, что и «{seed_ru}», и выпущенных в {y1}–{y2} годах.",
            query_text_en=f"Name {requested} smartphones manufactured by {maker_en}, running on at least one of the same operating systems as {seed_en}, and released between {y1} and {y2}.",
            constraints={"manufacturer": maker_en, "operating_system_from_model": seed_en, "year_from": y1, "year_to": y2},
            where_lines=_sp_where_hidden_os_maker_window(maker_qid, seed_qid, y1, y2), requested_count=requested,
            bridge_meta={"bridge": "operating_system", "constraint_key": "operating_system_from_model", "semantics": "shared_or_related_value", "intermediate_value_hidden_in_query": True, "wikidata_property": "P306", "seed_label_en": seed_en, "seed_label_ru": seed_ru},
        )

    if template == "l4_hidden_maker_year":
        # Same hidden maker bridge as L3, but levelled as L4 when used alongside a narrow 2-4 year window.
        cand = _sp_next_candidate("l4_hidden_maker_year", lambda: find_sp_hidden_maker_year_candidates(min_models=requested + 1, max_candidates_per_maker=14), rng)
        if not cand: return None
        maker_qid, maker_ru, maker_en, seed_qid, seed_ru, seed_en, y1, y2, _ = cand
        return _sp_finalize_wdqs_example(
            idx=idx, complexity=complexity,
            template_id="smartphones_l4_same_manufacturer_as_model_year_window", template_family="same_manufacturer_year_window",
            query_text_ru=f"Назови {requested} смартфонов, выпущенных тем же производителем, что и «{seed_ru}», в период {y1}–{y2} годов.",
            query_text_en=f"Name {requested} smartphones made by the same manufacturer as {seed_en} and released between {y1} and {y2}.",
            constraints={"manufacturer_from_model": seed_en, "year_from": y1, "year_to": y2},
            where_lines=_sp_where_hidden_maker_year(seed_qid, y1, y2), requested_count=requested,
            bridge_meta={"bridge": "manufacturer", "constraint_key": "manufacturer_from_model", "semantics": "shared_or_related_value", "intermediate_value_hidden_in_query": True, "wikidata_property": "P176", "seed_label_en": seed_en, "seed_label_ru": seed_ru},
        )

    if template == "l5_between_hidden":
        cand = _sp_next_candidate("l5_between_hidden", lambda: find_sp_l5_between_hidden_candidates(min_models_between=requested), rng)
        if not cand: return None
        (maker_qid, maker_ru, maker_en,
         seed_maker_qid, seed_maker_ru, seed_maker_en,
         seed_os_qid, seed_os_ru, seed_os_en, os_qid, os_ru, os_en,
         left_qid, left_ru, left_en, left_year,
         right_qid, right_ru, right_en, right_year,
         cnt) = cand
        return _sp_finalize_wdqs_example(
            idx=idx, complexity=complexity,
            template_id="smartphones_l5_same_manufacturer_same_os_between_models", template_family="same_manufacturer_same_os_between_references",
            query_text_ru=(
                f"Назови {requested} смартфона, выпущенных тем же производителем, что и «{seed_maker_ru}», "
                f"работающих хотя бы на одной из тех же операционных систем, что и «{seed_os_ru}», "
                f"выпущенных позже модели «{left_ru}» ({left_year}), но раньше модели «{right_ru}» ({right_year})."
            ),
            query_text_en=(
                f"Name {requested} smartphones made by the same manufacturer as {seed_maker_en}, "
                f"running on at least one of the same operating systems as {seed_os_en}, "
                f"and released after {left_en} ({left_year}) but before {right_en} ({right_year})."
            ),
            constraints={
                "manufacturer_from_model": seed_maker_en,
                "operating_system_from_model": seed_os_en,
                "released_after_model": left_en,
                "released_before_model": right_en,
                "left_reference_model_year": left_year,
                "right_reference_model_year": right_year,
            },
            where_lines=_sp_where_l5_between_hidden(seed_maker_qid, seed_os_qid, left_qid, right_qid),
            requested_count=requested,
            bridge_meta={
                "manufacturer_bridge": {"constraint_key": "manufacturer_from_model", "wikidata_property": "P176", "seed_label_en": seed_maker_en, "seed_label_ru": seed_maker_ru},
                "operating_system_bridge": {"constraint_key": "operating_system_from_model", "wikidata_property": "P306", "seed_label_en": seed_os_en, "seed_label_ru": seed_os_ru},
                "intermediate_values_hidden_in_query": True,
            },
        )


    if template == "l5_between_hidden_maker_direct_os":
        cand = _sp_next_candidate("l5_between_hidden_direct_os", lambda: find_sp_l5_between_hidden_candidates(min_models_between=requested), rng)
        if not cand: return None
        (maker_qid, maker_ru, maker_en,
         seed_maker_qid, seed_maker_ru, seed_maker_en,
         seed_os_qid, seed_os_ru, seed_os_en, os_qid, os_ru, os_en,
         left_qid, left_ru, left_en, left_year,
         right_qid, right_ru, right_en, right_year,
         cnt) = cand
        return _sp_finalize_wdqs_example(
            idx=idx, complexity=complexity,
            template_id="smartphones_l5_same_manufacturer_os_between_models", template_family="same_manufacturer_os_between_references",
            query_text_ru=(
                f"Назови {requested} смартфона, выпущенных тем же производителем, что и «{seed_maker_ru}», "
                f"работающих на {os_ru}, выпущенных позже модели «{left_ru}» ({left_year}), "
                f"но раньше модели «{right_ru}» ({right_year})."
            ),
            query_text_en=(
                f"Name {requested} smartphones made by the same manufacturer as {seed_maker_en}, "
                f"running on {os_en}, and released after {left_en} ({left_year}) but before {right_en} ({right_year})."
            ),
            constraints={
                "manufacturer_from_model": seed_maker_en,
                "operating_system": os_en,
                "released_after_model": left_en,
                "released_before_model": right_en,
                "left_reference_model_year": left_year,
                "right_reference_model_year": right_year,
            },
            where_lines=_sp_where_l5_between_hidden_maker_direct_os(seed_maker_qid, os_qid, left_qid, right_qid),
            requested_count=requested,
            bridge_meta={"bridge": "manufacturer", "constraint_key": "manufacturer_from_model", "semantics": "shared_or_related_value", "intermediate_value_hidden_in_query": True, "wikidata_property": "P176", "seed_label_en": seed_maker_en, "seed_label_ru": seed_maker_ru},
        )

    return None


_SMARTPHONES_TEMPLATES_BY_LEVEL = {
    "L1": ["l1_maker"],
    "L2": ["l2_maker_os", "l2_maker_year"],
    "L3": ["l3_maker_os_year", "l3_hidden_maker_year"],
    "L4": ["l4_hidden_os_maker_window", "l4_reference", "l4_hidden_maker_year"],
    "L5": ["l5_between_hidden", "l5_between_hidden_maker_direct_os"],
}


def _generate_smartphones_example_patched(complexity: str, idx: int, rng: random.Random, max_attempts: int = 80) -> BenchmarkExample:
    templates = list(_SMARTPHONES_TEMPLATES_BY_LEVEL.get(complexity, []))
    if not templates:
        raise ValueError(f"Unknown smartphones complexity: {complexity}")
    # Rotate the initial template by idx to avoid long runs of one family.
    offset = idx % len(templates)
    templates = templates[offset:] + templates[:offset]
    for attempt in range(max(1, int(max_attempts))):
        template = templates[attempt % len(templates)]
        ex = _sp_make_from_template(complexity, idx, rng, template)
        if ex is not None:
            return ex
    raise RuntimeError(f"Smartphones {complexity}: no valid example after {max_attempts} attempts")


def generate_smartphones_example_advanced(complexity: str, idx: int, rng: random.Random, max_attempts: int = 80) -> BenchmarkExample:
    return _generate_smartphones_example_patched(complexity, idx, rng, max_attempts=max_attempts)


def generate_smartphones_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 80) -> BenchmarkExample:
    return _generate_smartphones_example_patched(complexity, idx, rng, max_attempts=max_attempts)

print("Patched smartphones generator loaded: rich schema, clean constraints, WDQS ASK validators, L4/L5 multihop templates.")
